# 04 — LSTM / Seq2Seq
**Entrega 4.** Red Seq2Seq: el encoder lee 60 días de historia, el decoder genera 252 pasos directamente.

Instalar PyTorch (CPU): `pip install torch --index-url https://download.pytorch.org/whl/cpu`

In [8]:
import sys, warnings
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
warnings.filterwarnings("ignore")
sys.path.insert(0, ".")
from utils import load_data, compute_rmse, make_submission, train_val_split, INDEX_COLS

data       = load_data()
train_full = data["train_indices"][INDEX_COLS]
test_dates = data["test_dates"].index
train, val = train_val_split(train_full, val_size=252)
macro      = data.get("train_macro_factors")
network    = data.get("train_network_metrics")

WINDOW       = 60
HIDDEN       = 64
LAYERS       = 1
EPOCHS       = 15
BATCH        = 32
N_STEPS_VAL  = len(val)         # 252 — horizonte de validación
N_STEPS_TEST = len(test_dates)  # dinámico: 252, 262, etc.
print(f"PyTorch version: {torch.__version__}  |  N_STEPS_VAL={N_STEPS_VAL}  N_STEPS_TEST={N_STEPS_TEST}")


PyTorch version: 2.11.0+cu128  |  N_STEPS_VAL=252  N_STEPS_TEST=262


## Preparación de datos

In [9]:
def prepare_data(indices, macro=None, network=None):
    df = indices[INDEX_COLS].copy()
    if macro   is not None: df = pd.concat([df, macro.reindex(df.index).ffill().fillna(0)],   axis=1)
    if network is not None: df = pd.concat([df, network.reindex(df.index).ffill().fillna(0)], axis=1)
    df = df.ffill().fillna(0)
    scaler_idx = StandardScaler()
    scaler_all = StandardScaler()
    scaler_idx.fit(df[INDEX_COLS].values)
    scaled     = scaler_all.fit_transform(df.values)
    return scaled, scaler_idx

def make_sequences(values, window, horizon):
    X, y = [], []
    for i in range(window, len(values) - horizon + 1):
        X.append(values[i - window:i])
        y.append(values[i:i + horizon, :6])
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)

macro_tr = macro.iloc[:-252] if macro is not None else None
net_tr   = network.iloc[:-252] if network is not None else None
scaled_tr, scaler_idx = prepare_data(train, macro_tr, net_tr)
X_tr, y_tr = make_sequences(scaled_tr, WINDOW, N_STEPS_VAL)
print(f"X: {X_tr.shape}  y: {y_tr.shape}")


X: (1784, 60, 12)  y: (1784, 252, 6)


## Arquitectura Seq2Seq

In [10]:
class Encoder(nn.Module):
    def __init__(self, in_size, hidden, n_layers):
        super().__init__()
        self.lstm = nn.LSTM(in_size, hidden, n_layers,
                            batch_first=True, dropout=0.2)
    def forward(self, x):
        _, (h, c) = self.lstm(x)
        return h, c

class Decoder(nn.Module):
    def __init__(self, n_out, hidden, n_layers):
        super().__init__()
        self.lstm = nn.LSTM(n_out, hidden, n_layers,
                            batch_first=True, dropout=0.2)
        self.fc   = nn.Linear(hidden, n_out)
    def forward(self, h, c, steps):
        batch = h.shape[1]
        dec_in = torch.zeros(batch, 1, 6)
        outs = []
        for _ in range(steps):
            out, (h, c) = self.lstm(dec_in, (h, c))
            pred = self.fc(out)
            outs.append(pred)
            dec_in = pred
        return torch.cat(outs, dim=1)   # (batch, steps, 6)

n_features = X_tr.shape[2]
encoder = Encoder(n_features, HIDDEN, LAYERS)
decoder = Decoder(6, HIDDEN, LAYERS)
optimizer = torch.optim.Adam(
    list(encoder.parameters()) + list(decoder.parameters()), lr=1e-3)
loss_fn = nn.MSELoss()
print(f"Encoder params: {sum(p.numel() for p in encoder.parameters()):,}")
print(f"Decoder params: {sum(p.numel() for p in decoder.parameters()):,}")


Encoder params: 19,968
Decoder params: 18,822


## Entrenamiento

In [11]:
Xt = torch.tensor(X_tr)
yt = torch.tensor(y_tr)

for ep in range(EPOCHS):
    idx = np.random.permutation(len(Xt))
    total = 0.0
    for s in range(0, len(Xt), BATCH):
        b = idx[s:s + BATCH]
        optimizer.zero_grad()
        h, c  = encoder(Xt[b])
        out   = decoder(h, c, N_STEPS_VAL)
        loss  = loss_fn(out, yt[b])
        loss.backward()
        optimizer.step()
        total += loss.item() * len(b)
    if (ep + 1) % 10 == 0:
        print(f"  Epoch {ep+1:>3}/{EPOCHS}  loss={total/len(Xt):.6f}")


  Epoch  10/15  loss=0.184057


## Validación local

In [12]:
encoder.eval(); decoder.eval()
seed = torch.tensor(scaled_tr[-WINDOW:][None], dtype=torch.float32)
with torch.no_grad():
    h, c = encoder(seed)
    raw  = decoder(h, c, N_STEPS_VAL).numpy()[0]   # (N_STEPS_VAL, 6)

preds_val = scaler_idx.inverse_transform(raw)
pred_df   = pd.DataFrame(preds_val, index=val.index, columns=INDEX_COLS)
rmse = compute_rmse(val, pred_df)
print(f"[Seq2Seq] RMSE local = {rmse:,.2f}")
per = np.sqrt(((val.values - pred_df.values)**2).mean(axis=0))
for col, r in zip(INDEX_COLS, per):
    print(f"  {col}: {r:,.2f}")


[Seq2Seq] RMSE local = 3,055.57
  Index_A: 3,293.84
  Index_B: 818.92
  Index_C: 7.53
  Index_D: 3,382.05
  Index_E: 21.22
  Index_F: 10,809.88


## Generar submission

In [13]:
print("Entrenando en datos completos ...")
# Escalamos train_full pero reutilizamos scaler_idx (fiteado solo en train) para
# la inverse_transform de test — evita que estadísticas de val contaminen test.
scaled_full, _scaler_unused = prepare_data(train_full, macro, network)
X_full, y_full = make_sequences(scaled_full, WINDOW, N_STEPS_TEST)

enc_f = Encoder(X_full.shape[2], HIDDEN, LAYERS)
dec_f = Decoder(6, HIDDEN, LAYERS)
opt_f = torch.optim.Adam(
    list(enc_f.parameters()) + list(dec_f.parameters()), lr=1e-3)
Xf = torch.tensor(X_full); yf = torch.tensor(y_full)
for ep in range(EPOCHS):
    idx = np.random.permutation(len(Xf))
    for s in range(0, len(Xf), BATCH):
        b = idx[s:s + BATCH]
        opt_f.zero_grad()
        h, c = enc_f(Xf[b])
        loss_fn(dec_f(h, c, N_STEPS_TEST), yf[b]).backward()
        opt_f.step()
    if (ep + 1) % 10 == 0:
        print(f"  Epoch {ep+1}/{EPOCHS}")

enc_f.eval(); dec_f.eval()
seed_f = torch.tensor(scaled_full[-WINDOW:][None], dtype=torch.float32)
with torch.no_grad():
    h, c  = enc_f(seed_f)
    raw_f = dec_f(h, c, N_STEPS_TEST).numpy()[0]   # (N_STEPS_TEST, 6)

# scaler_idx fue fiteado solo sobre train (celda b02ed859) — correcto para test
preds_test = scaler_idx.inverse_transform(raw_f)
pred_test  = pd.DataFrame(preds_test, index=test_dates, columns=INDEX_COLS)
make_submission(pred_test, "submission_04_lstm_seq2seq.csv")
pred_test.head()

Entrenando en datos completos ...
  Epoch 10/15
Submission saved: c:\Users\1jose\Desktop\previa hackatlon\hackathon\submissions\submission_04_lstm_seq2seq.csv


,Index_A,Index_B,Index_C,Index_D,Index_E,Index_F
Date,,,,,,
2024-01-01,11593.468750,3734.332275,18.413530,11642.000977,100.905212,27468.562500
2024-01-02,11943.253906,3798.795898,19.229322,12102.920898,102.314903,28758.128906
2024-01-03,12146.849609,3850.394531,19.747915,12365.074219,103.409416,29065.634766
2024-01-04,12100.700195,3886.035889,20.086378,12294.624023,103.846367,28321.896484
2024-01-05,11918.273438,3907.988525,20.357382,12063.973633,104.014267,27008.091797
